# 12. Customer Action Table (Deliverable)

Strategy 8.3-8.4. Takes the modeling team's scores
(`churn_probability_calibrated`, `churn_model_h90_calibrated`), adds customer
value `value_v`, RFM segment and the Strategy v2 decision rule, and produces:

1. the **customer-level score table with actions** (Retain / Nurture / No-offer),
2. its realized margin on the **test** weeks (scenario estimate),
3. the **current target list** for the last scored Monday (unlabeled) and the dashboard KPIs.

The calibrated probability is used as delivered. `expected_loss` from the same
file is not used: it is not an incremental-margin quantity (it ignores the
discount given to customers who would have bought anyway); a check in the
analysis showed it ranks no better than the recency rule.

In [1]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from policy_lib import *

SCEN = dict(d=0.10, e=0.30, m=0.40, r=0.40, c=0.50)       # Base scenario (Strategy 7.8)
SC = (SCEN["d"], SCEN["e"], SCEN["m"], SCEN["r"], SCEN["c"])
K = 0.20                                                  # operating target share
NURTURE_PCT = 0.80                                        # top 20% risk, not selected -> low-cost contact
MODEL_VERSION = "lr_extended_c0.01_weighted_cal_h90"

data = pd.read_parquet("../data/processed/modeling_dataset_h90.parquet")
scores = pd.read_parquet("../data/results/churn_scores_h90_calibrated.parquet")
key = ["customer_id", "snapshot_date"]
assert not scores.duplicated(key).any()

## 12.1 Join and integrity checks

In [2]:
feat = data[key + ["value_v", "rfm_segment", "recency_days", "segment"]].rename(columns={"segment": "customer_type"})
t = scores.merge(feat, on=key, how="left", validate="one_to_one", indicator=True)
# Deployment weeks (after the last labelable week) have no row in the labelable dataset only if absent from it
print(t["_merge"].value_counts().to_dict())
print(t.groupby("split").agg(rows=("customer_id", "size"), weeks=("snapshot_date", "nunique"),
      missing_value=("value_v", lambda s: s.isna().sum()), missing_rfm=("rfm_segment", lambda s: s.isna().sum())))
assert t.value_v.notna().all() and t.rfm_segment.notna().all()
assert t.model_version.nunique() == 1 and t.model_version.iloc[0] == MODEL_VERSION
assert t.churn_probability_calibrated.between(0, 1).all()
t = t.drop(columns="_merge")

{'both': 60455, 'left_only': 0, 'right_only': 0}
             rows  weeks  missing_value  missing_rfm
split                                               
deployment  36203     13              0            0
test        24252     10              0            0


## 12.2 Decision rule

* `expected_margin` = E[ΔM] = p·e·V·(m−d) − (1−p)·r·V·d − c, with the calibrated `p`.
* **Retain**: within the weekly top-K by `expected_margin` and `expected_margin` > 0.
* **Nurture**: not Retain but in the top 20% of the week's calibrated risk
  (at risk, but an offer would not pay): low-cost contact, no discount.
* **No-offer**: the rest (likely to buy anyway, or low expected value).

In [3]:
def add_actions(df, k=K, scen=SC):
    d, e, m, r, c = scen
    df = df.copy()
    df["expected_margin"] = expected_margin(df.churn_probability_calibrated, df.value_v, d, e, m, r, c)
    df["rank_by_margin"] = df.groupby("snapshot_date").expected_margin.rank(ascending=False, method="first").astype(int)
    retain = top_k_mask(df, "expected_margin", k) & (df.expected_margin > 0)
    risky = df.groupby("snapshot_date").churn_probability_calibrated.rank(pct=True) >= NURTURE_PCT
    df["action"] = np.where(retain, "Retain", np.where(risky, "Nurture", "No-offer"))
    return df

t = add_actions(t)
t.groupby(["split", "action"]).agg(rows=("customer_id", "size"), mean_p=("churn_probability_calibrated", "mean"),
                                   mean_value=("value_v", "mean"), churn=("churn", "mean")).round(3)

rows  mean_p  mean_value  churn
split      action                                    
deployment No-offer  25864   0.254     383.875    NaN
           Nurture    3099   0.676     128.848    NaN
           Retain     7240   0.600     491.500    NaN
test       No-offer  17358   0.267     377.187  0.305
           Nurture    2044   0.698     122.240  0.721
           Retain     4850   0.623     467.141  0.608

## 12.3 Test-period evaluation (labelled weeks)

Policies on the same test weeks with the calibrated probability. Realized
scenario margin uses the actual labels; K=20%, Base scenario. £ per week.

In [4]:
test = t[t.split == "test"].copy()
test["rand"] = np.random.default_rng(0).uniform(size=len(test))
masks = {
    "No offer (P0)": pd.Series(False, index=test.index),
    "Blanket (P1)": pd.Series(True, index=test.index),
    "Random (P2)": top_k_mask(test, "rand", K),
    "Recency (P3)": top_k_mask(test, "recency_days", K),
    "Risk-only, calibrated p (P5)": top_k_mask(test, "churn_probability_calibrated", K),
    "Value-aware, E[dM] > 0 (P6) = Retain": test.action == "Retain",
}
rows = {}
for n, mk in masks.items():
    w = weekly_margin(test, mk, SC)
    rows[n] = dict(contacts_week=mk.groupby(test.snapshot_date).sum().mean(), margin_week=w.mean(), std=w.std(),
                   worst_week=w.min(), share_to_would_be_buyers=(1 - test[mk].churn.mean()) if mk.any() else np.nan,
                   margin_per_contact=w.sum() / max(mk.sum(), 1))
policy_test = pd.DataFrame(rows).T
policy_test.to_csv("../data/results/test_policy_comparison.csv")
display(policy_test.round(2))

,contacts_week,margin_week,std,worst_week,share_to_would_be_buyers,margin_per_contact
No offer (P0),0.0,0.00,0.00,0.00,NaN,0.00
Blanket (P1),2425.2,4419.86,3265.61,-1036.92,0.60,1.82
Random (P2),485.0,765.27,823.11,-617.23,0.60,1.58
Recency (P3),485.0,4910.80,1051.94,2933.84,0.42,10.13
"Risk-only, calibrated p (P5)",485.0,5765.20,629.25,4731.14,0.31,11.89
"Value-aware, E[dM] > 0 (P6) = Retain",485.0,7991.23,1113.89,6269.13,0.39,16.48


In [5]:
# Same, by action group: where does the Retain margin come from? (per customer, realized, base scenario)
test["gain"] = row_gain(test, *SC)
test.groupby("action").agg(rows=("gain", "size"), churn_rate=("churn", "mean"), mean_value=("value_v", "mean"),
                           realized_margin_per_contact=("gain", "mean")).round(2)

,rows,churn_rate,mean_value,realized_margin_per_contact
action,,,,
No-offer,17358,0.30,377.19,-2.75
Nurture,2044,0.72,122.24,5.90
Retain,4850,0.61,467.14,16.48


In [6]:
# Retain composition by RFM segment (test)
comp = test.groupby(["action", "rfm_segment"]).size().unstack(0).fillna(0)
(comp / comp.sum()).round(3)

action,No-offer,Nurture,Retain
rfm_segment,,,
At Risk,0.099,0.161,0.345
Can't Lose,0.217,0.000,0.052
Champions,0.384,0.000,0.005
Hibernating,0.006,0.505,0.360
Loyal,0.189,0.000,0.014
Potential,0.106,0.334,0.225


## 12.4 Current target list and dashboard KPIs

The last scored Monday has no outcome yet (unlabelled): this is the list marketing would act on.

In [7]:
last = t[t.snapshot_date == t.snapshot_date.max()].sort_values("rank_by_margin")
print("Scoring date:", last.snapshot_date.iloc[0].date(), "| active customers:", len(last))
kpi = pd.Series({
    "Active customers": len(last),
    "Retain (offer)": int((last.action == "Retain").sum()),
    "Nurture (no discount)": int((last.action == "Nurture").sum()),
    "No-offer": int((last.action == "No-offer").sum()),
    "Revenue at risk in Retain list (sum of V x p, £)": float((last[last.action == "Retain"].value_v * last[last.action == "Retain"].churn_probability_calibrated).sum()),
    "Expected incremental margin of Retain list (Base, £)": float(last[last.action == "Retain"].expected_margin.sum()),
    "Expected share of Retain offers to would-be buyers": float(1 - last[last.action == "Retain"].churn_probability_calibrated.mean()),
})
display(kpi.round(3).to_frame("value"))
last[["customer_id", "rfm_segment", "customer_type", "churn_probability_calibrated", "value_v", "expected_margin", "action"]].head(15).round(3)

Scoring date: 2011-12-05 | active customers: 3181


,value
Active customers,3181.000
Retain (offer),636.000
Nurture (no discount),247.000
No-offer,2298.000
"Revenue at risk in Retain list (sum of V x p, £)",169445.968
"Expected incremental margin of Retain list (Base, £)",9889.180
Expected share of Retain offers to would-be buyers,0.398


,customer_id,rfm_segment,customer_type,churn_probability_calibrated,value_v,expected_margin,action
57687,12378,At Risk,repeat,0.628,1956.840,80.963,Retain
57543,16902,At Risk,one-time,0.663,1706.880,78.248,Retain
57469,13850,At Risk,one-time,0.686,1595.480,77.922,Retain
57863,12688,At Risk,one-time,0.586,1956.840,70.276,Retain
58008,14887,At Risk,one-time,0.555,1862.000,59.429,Retain
57951,15947,At Risk,one-time,0.567,1708.240,57.056,Retain
57401,12534,Hibernating,one-time,0.717,981.180,51.754,Retain
58092,16380,At Risk,one-time,0.529,1743.150,49.739,Retain
58192,13422,At Risk,one-time,0.505,1956.840,49.582,Retain
57393,13075,Hibernating,one-time,0.720,919.610,48.778,Retain


## 12.5 Export

In [8]:
cols = ["customer_id", "snapshot_date", "split", "churn_probability", "churn_probability_calibrated", "risk_score",
        "risk_band", "value_v", "rfm_segment", "customer_type", "expected_margin", "rank_by_margin", "action",
        "model_version", "churn"]
out = t[cols].rename(columns={"churn": "churn_actual"})
out["scenario"] = "Base d=10% e=30% m=40% r=40% c=0.50; K=20%"
out.to_parquet("../data/results/customer_scores_actions_h90.parquet", index=False)
print(out.shape, "-> data/results/customer_scores_actions_h90.parquet")

(60455, 16) -> data/results/customer_scores_actions_h90.parquet
